# Task 2A — Extract Pixel Values (Inside Boundary) to CSV

Converted from `Task_2A_Pixel_Values.py` on 2025-12-29 08:35:08.

Run **top-to-bottom**. Cells are broken into steps so you can **see outputs at every stage**.

## 1) Library imports

In [2]:
import os
import numpy as np
import pandas as pd

import rasterio
from rasterio.warp import reproject, Resampling

print("All imports OK.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("rasterio:", rasterio.__version__)


All imports OK.
numpy: 1.26.4
pandas: 2.3.3
rasterio: 1.4.4


## 2) User inputs (Edit this cell)

In [8]:
folders = {
    "AET":  r"E:\VUB\Final\AET_Clipped",
    "LULC": r"E:\VUB\Final\LULC_Clipped",
    "P":    r"E:\VUB\Final\Precipitation_Clipped",
    "RZSM": r"E:\VUB\Final\RootZoneSoilMoisture_Clipped",
    "TEMP": r"E:\VUB\Final\Temperature_Mean_Clipped",
}

SOIL_FILE = r"E:\VUB\Final\Soil_Clipped\Soil_HSG_10km_clipped.tif"

YEARS = range(2014, 2025)  # 2014..2023
OUT_CSV = r"E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv"

CATEGORICAL = {"LULC", "SOIL"}
ZERO_AS_NODATA = {"LULC", "SOIL"}

print("Folders:")
for k, v in folders.items():
    print(f"  {k}: {v}")
print("SOIL_FILE:", SOIL_FILE)
print("YEARS:", list(YEARS)[:3], "...", list(YEARS)[-3:])
print("OUT_CSV:", OUT_CSV)


Folders:
  AET: E:\VUB\Final\AET_Clipped
  LULC: E:\VUB\Final\LULC_Clipped
  P: E:\VUB\Final\Precipitation_Clipped
  RZSM: E:\VUB\Final\RootZoneSoilMoisture_Clipped
  TEMP: E:\VUB\Final\Temperature_Mean_Clipped
SOIL_FILE: E:\VUB\Final\Soil_Clipped\Soil_HSG_10km_clipped.tif
YEARS: [2014, 2015, 2016] ... [2022, 2023, 2024]
OUT_CSV: E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv


## 3) Missing data checks

In [9]:
missing = False

for var, folder in folders.items():
    if not os.path.isdir(folder):
        print("❌ Missing folder:", var, "->", folder)
        missing = True
    else:
        tifs = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]
        print(f"✅ {var}: {len(tifs)} tif(s) in {folder}")

if not os.path.exists(SOIL_FILE):
    print("❌ Missing SOIL_FILE:", SOIL_FILE)
    missing = True
else:
    print("✅ SOIL_FILE exists")

out_dir = os.path.dirname(OUT_CSV)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
    print("✅ Output directory ready:", out_dir)

if missing:
    raise FileNotFoundError("Fix missing paths in the Config cell and rerun.")


✅ AET: 11 tif(s) in E:\VUB\Final\AET_Clipped
✅ LULC: 11 tif(s) in E:\VUB\Final\LULC_Clipped
✅ P: 11 tif(s) in E:\VUB\Final\Precipitation_Clipped
✅ RZSM: 11 tif(s) in E:\VUB\Final\RootZoneSoilMoisture_Clipped
✅ TEMP: 11 tif(s) in E:\VUB\Final\Temperature_Mean_Clipped
✅ SOIL_FILE exists
✅ Output directory ready: E:\VUB\Final\PixelDataFrames


## 4) Helper functions

In [10]:
def find_year_tif(folder, year):
    y = str(year)
    for f in os.listdir(folder):
        if f.lower().endswith(".tif") and y in f:
            return os.path.join(folder, f)
    return None

def window_coords_pixelid(ref_ds, win):
    W = ref_ds.width
    rows = np.arange(win.row_off, win.row_off + win.height, dtype=np.int32)
    cols = np.arange(win.col_off, win.col_off + win.width, dtype=np.int32)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")

    pixel_id = (rr.astype(np.int64) * np.int64(W) + cc.astype(np.int64)).ravel()
    xs, ys = rasterio.transform.xy(ref_ds.transform, rr, cc, offset="center")
    lon = np.asarray(xs, dtype=np.float64).ravel()
    lat = np.asarray(ys, dtype=np.float64).ravel()
    return rr.ravel(), cc.ravel(), lon, lat, pixel_id

def read_aligned_window(src_ds, ref_ds, win, layer_name, is_categorical):
    # Fast path: already aligned
    if (src_ds.crs == ref_ds.crs and
        src_ds.transform == ref_ds.transform and
        src_ds.width == ref_ds.width and
        src_ds.height == ref_ds.height):
        arr = src_ds.read(1, window=win).astype(np.float32)
        nodata = src_ds.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)
        if layer_name in ZERO_AS_NODATA:
            arr = np.where(arr == 0, np.nan, arr)
        return arr

    # Reproject to ref window
    dst_transform = rasterio.windows.transform(win, ref_ds.transform)
    dst_h, dst_w = win.height, win.width
    dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)

    resamp = Resampling.nearest if is_categorical else Resampling.bilinear

    reproject(
        source=rasterio.band(src_ds, 1),
        destination=dst,
        src_transform=src_ds.transform,
        src_crs=src_ds.crs,
        dst_transform=dst_transform,
        dst_crs=ref_ds.crs,
        src_nodata=src_ds.nodata,
        dst_nodata=np.nan,
        resampling=resamp
    )

    if layer_name in ZERO_AS_NODATA:
        dst = np.where(dst == 0, np.nan, dst)

    return dst

print("Helper functions loaded.")


Helper functions loaded.


## 5) Preview file discovery for a sample year

In [11]:
SAMPLE_YEAR = list(YEARS)[0]
print("Sample year:", SAMPLE_YEAR)

for var, folder in folders.items():
    fp = find_year_tif(folder, SAMPLE_YEAR)
    print(f"{var}: {fp if fp else '❌ not found'}")


Sample year: 2014
AET: E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif
LULC: E:\VUB\Final\LULC_Clipped\LULC_2014_10Km_clipped.tif
P: E:\VUB\Final\Precipitation_Clipped\Precipitation_2014_clipped.tif
RZSM: E:\VUB\Final\RootZoneSoilMoisture_Clipped\RZSM_2014_clipped.tif
TEMP: E:\VUB\Final\Temperature_Mean_Clipped\Temperature_Mean_2014_clipped.tif


## 6) Run extraction (writes one growing CSV)

In [12]:
soil_ds = rasterio.open(SOIL_FILE)

if os.path.exists(OUT_CSV):
    os.remove(OUT_CSV)
    print("Removed existing OUT_CSV:", OUT_CSV)

first_write = True

for year in YEARS:
    print(f"\n=== Year {year} ===")

    year_files = {}
    for var, folder in folders.items():
        fp = find_year_tif(folder, year)
        if fp is None:
            raise FileNotFoundError(f"Missing {var} tif for year {year} in: {folder}")
        year_files[var] = fp

    with rasterio.open(year_files["AET"]) as ref:
        open_year = {v: rasterio.open(p) for v, p in year_files.items()}

        try:
            chunks = []
            win_count = 0
            inside_pix_total = 0

            for _, win in ref.block_windows(1):
                win_count += 1
                aet_masked = ref.read(1, window=win, masked=True)
                inside = (~aet_masked.mask).ravel()

                if not inside.any():
                    continue

                inside_pix_total += int(inside.sum())

                row, col, lon, lat, pixel_id = window_coords_pixelid(ref, win)

                data = {
                    "year": np.full(inside.sum(), year, dtype=np.int16),
                    "pixel_id": pixel_id[inside],
                    "row": row[inside],
                    "col": col[inside],
                    "lon": lon[inside],
                    "lat": lat[inside],
                }

                for var, ds in open_year.items():
                    arr = read_aligned_window(ds, ref, win, layer_name=var, is_categorical=(var in CATEGORICAL))
                    data[var] = arr.ravel()[inside]

                soil_arr = read_aligned_window(soil_ds, ref, win, layer_name="SOIL", is_categorical=True)
                data["SOIL"] = soil_arr.ravel()[inside]

                chunks.append(pd.DataFrame(data))

            print("Windows scanned:", win_count)
            print("Inside pixels processed:", inside_pix_total)

            if not chunks:
                print("No inside pixels found for year:", year)
                continue

            df_y = pd.concat(chunks, ignore_index=True)
            predictor_cols = list(year_files.keys()) + ["SOIL"]
            df_y = df_y.dropna(subset=predictor_cols, how="all")

            df_y.to_csv(OUT_CSV, index=False, mode="w" if first_write else "a", header=first_write)
            first_write = False

            print("Appended rows:", len(df_y))
            display(df_y.head(5))

        finally:
            for ds in open_year.values():
                ds.close()

soil_ds.close()

print("\nDONE. Saved:", OUT_CSV)


Removed existing OUT_CSV: E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv

=== Year 2014 ===
Windows scanned: 355
Inside pixels processed: 55929
Appended rows: 55929


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2014,382,0,382,27.55,70.05,204.139893,NaN,238.719223,0.270068,-0.036160,4.0
1,2014,383,0,383,27.65,70.05,206.531601,30.0,233.095627,0.270511,-0.002954,4.0
2,2014,384,0,384,27.75,70.05,201.743500,30.0,229.782440,0.293638,0.038191,4.0
3,2014,385,0,385,27.85,70.05,213.141602,30.0,229.219025,0.272055,0.080452,4.0
4,2014,386,0,386,27.95,70.05,228.954285,30.0,230.524460,0.347064,0.099918,4.0



=== Year 2015 ===
Windows scanned: 355
Inside pixels processed: 55929
Appended rows: 55929


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2015,382,0,382,27.55,70.05,192.765884,NaN,283.833099,0.266248,-0.005764,4.0
1,2015,383,0,383,27.65,70.05,189.677521,30.0,279.497864,0.272314,0.028190,4.0
2,2015,384,0,384,27.75,70.05,196.178314,30.0,276.411407,0.301021,0.072577,4.0
3,2015,385,0,385,27.85,70.05,233.507034,30.0,274.917175,0.271885,0.119786,4.0
4,2015,386,0,386,27.95,70.05,220.404587,30.0,274.495972,0.348186,0.144441,4.0



=== Year 2016 ===
Windows scanned: 355
Inside pixels processed: 55929
Appended rows: 55929


...


=== Year 2023 ===
Windows scanned: 355
Inside pixels processed: 55929
Appended rows: 55929


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2023,382,0,382,27.55,70.05,190.098190,NaN,263.936920,0.270171,-0.043037,4.0
1,2023,383,0,383,27.65,70.05,188.133972,30.0,264.616272,0.268882,-0.027659,4.0
2,2023,384,0,384,27.75,70.05,188.496323,30.0,265.914429,0.290290,0.003592,4.0
3,2023,385,0,385,27.85,70.05,235.973969,30.0,268.100098,0.336821,0.045676,4.0
4,2023,386,0,386,27.95,70.05,235.559738,30.0,270.513611,0.355861,0.068146,4.0



=== Year 2024 ===
Windows scanned: 355
Inside pixels processed: 55663
Appended rows: 55663


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2024,382,0,382,27.55,70.05,206.060135,NaN,266.398651,0.267576,0.429609,4.0
1,2024,383,0,383,27.65,70.05,206.829910,30.0,265.444885,0.268555,0.443021,4.0
2,2024,384,0,384,27.75,70.05,205.433411,30.0,265.944153,0.290443,0.469696,4.0
3,2024,385,0,385,27.85,70.05,259.412445,30.0,268.232727,0.337213,0.503007,4.0
4,2024,386,0,386,27.95,70.05,256.214844,30.0,271.458710,0.356767,0.517963,4.0



DONE. Saved: E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv


## 7) Verify output CSV 

In [13]:
if os.path.exists(OUT_CSV):
    df = pd.read_csv(OUT_CSV, nrows=200000)
    print("Sample rows loaded:", len(df))
    print("Columns:", list(df.columns))
    display(df.head(10))
else:
    print("OUT_CSV not found yet. Run the extraction cell first.")


Sample rows loaded: 200000
Columns: ['year', 'pixel_id', 'row', 'col', 'lon', 'lat', 'AET', 'LULC', 'P', 'RZSM', 'TEMP', 'SOIL']


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2014,382,0,382,27.55,70.05,204.13990,NaN,238.71922,0.270068,-0.036160,4.0
1,2014,383,0,383,27.65,70.05,206.53160,30.0,233.09563,0.270511,-0.002954,4.0
2,2014,384,0,384,27.75,70.05,201.74350,30.0,229.78244,0.293638,0.038191,4.0
3,2014,385,0,385,27.85,70.05,213.14160,30.0,229.21902,0.272055,0.080452,4.0
4,2014,386,0,386,27.95,70.05,228.95428,30.0,230.52446,0.347064,0.099918,4.0
5,2014,826,1,373,26.65,69.95,247.35196,30.0,262.63400,0.258783,-0.309810,4.0
6,2014,827,1,374,26.75,69.95,248.69392,30.0,259.89142,0.277007,-0.232099,NaN
7,2014,828,1,375,26.85,69.95,242.82750,30.0,256.99430,0.272365,-0.171541,4.0
8,2014,833,1,380,27.35,69.95,218.71545,30.0,485.81534,0.258491,-0.224810,4.0
9,2014,834,1,381,27.45,69.95,201.59355,30.0,537.55870,0.289886,-0.236375,4.0
